In [ ]:
import os, h5py
from sys import argv
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
def norm_scale_influe(influe, clip_value=2):
    adata = sc.AnnData(influe.copy())

    sc.pp.normalize_total(adata, target_sum=10)
    sc.pp.log1p(adata)
    sc.pp.scale(adata, max_value=np.inf)
    
    influe = adata.to_df().copy()
    if clip_value:
        influe = np.clip(influe, -clip_value, clip_value)

    return influe


def binary_influ(bin_influ_df, t=0.8):
    bin_influ_df = bin_influ_df.copy()

    t = np.quantile(bin_influ_df.values, t, axis=0)
    t[t < 0] = 0
    print(max(t), min(t))

    bin_influ_df[bin_influ_df <= t] = 0
    bin_influ_df[bin_influ_df > t] = 1
    bin_influ_df = bin_influ_df[bin_influ_df.sum(1) > 10]

    return bin_influ_df


def draw_influ_clustermap(bin_influ_df, anno, show_fig=True,
                          col_cluster=False, cmap='vlag', 
                          save_prefix="positive_influence", figtype='pdf',
                          save_dendrogram=False): 
    #import seaborn as sns
    #import matplotlib.pyplot as plt
    #from matplotlib.patches import Patch
    
    # clustermap
    color = ("#E6AB02", "#E41A1C", "#66A61E", "#D95F02", "#1B9E77", "#E7298A",  "#E31A1C", "#A6761D"  , "#B2DF8A",   "#FFFF99",   "#7570B3", "#FF7F00",  "#A65628", "#B3CDE3", "#BC80BD",     "#A6CEE3","#984EA3",   "#CCEBC5",  "#E41A1C",    "#4DAF4A","#BEBADA", "#B3DE69", "#CAB2D6","#FFFFB3",   "#33A02C","#B15928", "#6A3D9A","#FBB4AE",    "blue",          "#FB8072",      "#FFFF33","#CCEBC5",      "#A6761D",   "#2c7fb8","#fa9fb5",  "#BEBADA","#E7298A", "#E7298A", "green", "orange", "lightblue", "#BEBADA", "#33A02C", "#E31A1C", "#E6AB02", "#FFFF33", "lightblue", "#BC80BD", "#CCEBC5")
    regions = ("Secretory", "Germline", "Muscle", "Neuron" , "Immune", "Epithelial", "Glia", "Proliferating","Other",  "Neoblast","Protonephridia","Phagocytes","Cathepsin","Rectum", "Coelomocytes","Intestine","Hepatocyte","Pharynx","Endothelial","Erythroid","Testis","Mesenchyme","Yolk", "Midgut" ,"Embryo","Hemocytes",  "Fat",  "Unknown","Gastrodermis","DigFilaments","Pigment","BasementMembrane","Endoderm","RP_high","FatBody","Male","Nephron", "Pancreatic", "Neuroendocrine", "DigestiveGland", "Germ", "Stromal", "Non-seam", "Pharyn", "Precursors", "Seam", "Follicle", "MAG", "Notochord")
    color_regions = {x:y for x,y in zip(regions, color)}

    anno["colors_lineage"] = anno[['Cellcluster']].applymap(lambda x: color_regions[x])
    anno_color = anno.loc[bin_influ_df.index]
    
    lut = {cluster:color_regions.get(cluster) for cluster in anno.Cellcluster.unique()}
    handles = [Patch(facecolor=lut[name]) for name in lut]
    # print(lut)
    
    plt.figure(figsize=(6, 10))
    g = sns.clustermap(bin_influ_df.T, col_cluster=col_cluster,
                       cbar_pos=None, xticklabels=False,
                       col_colors=anno_color[["colors_lineage"]],
                       cmap=cmap, figsize=(18, 30),
                       dendrogram_ratio=(.01, .1), colors_ratio=0.01)

    plt.legend(handles, lut, title='CellLieange',
               bbox_to_anchor=(1, 1), bbox_transform=plt.gcf().transFigure, loc='upper right')

    plt.savefig(save_prefix+".clustermap."+figtype)
    if show_fig:
        plt.show()
    plt.close()
    
    if save_dendrogram and col_cluster:
        anno_color.iloc[g.dendrogram_col.reordered_ind,].to_csv(save_prefix+".cellanno.csv")
        print("saving dendrogram")


def melt_influ(bin_influ_df, Var="Celltype", t=None):
    bin_influ_df = bin_influ_df.copy()
    
    bin_influ_df['Var'] = bin_influ_df.index
    bin_influ_df = bin_influ_df.melt(id_vars='Var')
    bin_influ_df.columns = [Var, "Motif", "Influe"]
    
    if t:
        bin_influ_df = bin_influ_df[bin_influ_df.value > t]
        
    return bin_influ_df


def correlation_ratio(categories, measurements):
    fcat, _ = pd.factorize(categories)
    cat_num = np.max(fcat)+1
    y_avg_array = np.zeros(cat_num)
    n_array = np.zeros(cat_num)
    for i in range(0,cat_num):
        cat_measures = measurements[np.argwhere(fcat == i).flatten()]
        n_array[i] = len(cat_measures)
        y_avg_array[i] = np.average(cat_measures)
    y_total_avg = np.sum(np.multiply(y_avg_array,n_array))/np.sum(n_array)
    numerator = np.sum(np.multiply(n_array,np.power(np.subtract(y_avg_array,y_total_avg),2)))
    denominator = np.sum(np.power(np.subtract(measurements,y_total_avg),2))
    if numerator == 0:
        eta = 0.0
    else:
        eta = numerator/denominator
    return eta


def celltype_influ_analysis(influence, prefix):

    influence = norm_scale_influe(influence, clip_value=2)

    # anno_cell
    draw_influ_clustermap(influence, anno, show_fig=False, 
                          cmap='vlag', col_cluster=False, save_dendrogram=False,
                          save_prefix=prefix, figtype='png')
    draw_influ_clustermap(influence, anno, show_fig=False, 
                          cmap='vlag', col_cluster=True, save_dendrogram=True,
                          save_prefix=prefix+"_dendrogram", figtype='png')

    bin_influ = binary_influ(influence)
    draw_influ_clustermap(bin_influ, anno, show_fig=False, 
                            cmap='Greys', col_cluster=False, save_dendrogram=False,
                            save_prefix=prefix+"_bin", figtype='png')
    draw_influ_clustermap(bin_influ, anno, show_fig=False, 
                          cmap='Greys', col_cluster=True, save_dendrogram=True,
                          save_prefix=prefix+"_bin_dendrogram", figtype='png')

    # anno_celltype
    anno_celltype = anno_color.drop_duplicates(["Celltype", "Cellcluster"]).set_index("Celltype")
    influ_celltype = influence.groupby(anno_color.Celltype).mean()
    melt_influ(influ_celltype).to_csv(prefix+"_celltype.csv")

    draw_influ_clustermap(influ_celltype, anno_celltype, show_fig=False, 
                          cmap='vlag', col_cluster=False, save_dendrogram=False,
                          save_prefix=prefix+"_celltype", figtype='png')
    draw_influ_clustermap(influ_celltype, anno_celltype, show_fig=False, 
                          cmap='vlag', col_cluster=True, save_dendrogram=True,
                          save_prefix=prefix+"_celltype_dendrogram", figtype='png')


    bin_influ_celltype = binary_influ(influ_celltype)
    melt_influ(bin_influ_celltype).to_csv(prefix+"_celltype_bin.csv")
                    
    draw_influ_clustermap(bin_influ_celltype, anno_celltype, show_fig=False,
                            cmap='Greys', col_cluster=False, save_dendrogram=False,
                            save_prefix=prefix+"_celltype_bin", figtype='png')
    draw_influ_clustermap(bin_influ_celltype, anno_celltype, show_fig=False,
                          cmap='Greys', col_cluster=True, save_dendrogram=True,
                          save_prefix=prefix+"_celltype_bin_dendrogram", figtype='png')

    # anno_cluster
    anno_cluster = anno_color.drop_duplicates(["Cellcluster"])
    anno_cluster.index = anno_cluster.Cellcluster
    influ_cluster = influence.groupby(anno_color.Cellcluster).mean()
    melt_influ(influ_cluster, Var="Cluster").to_csv(prefix+"_cluster.csv")

    draw_influ_clustermap(influ_cluster, anno_cluster, show_fig=False,
                            cmap='vlag', col_cluster=True,
                            save_prefix=prefix+"_dendrogram")



In [ ]:
cellanno_atac = ['P53_1', 'P53_10', 'P53_11', 'P53_12', 'P53_13', 'P53_14',
       'P53_15', 'P53_16', 'P53_17', 'P53_18', 'P53_19', 'P53_2',
       'P53_20', 'P53_22', 'P53_24', 'P53_25', 'P53_3', 'P53_4', 'P53_5',
       'P53_6', 'P53_7', 'P53_8', 'P53_9', 'WT_1', 'WT_10', 'WT_11',
       'WT_12', 'WT_13', 'WT_15', 'WT_2', 'WT_20', 'WT_21', 'WT_22',
       'WT_23', 'WT_24', 'WT_25', 'WT_3', 'WT_4', 'WT_5', 'WT_6', 'WT_7',
       'WT_8', 'WT_9']


cellanno_rna = ['P53_0', 'P53_1', 'P53_10', 'P53_100', 'P53_101', 'P53_102',
  'P53_103', 'P53_104', 'P53_105', 'P53_106', 'P53_107', 'P53_108',
  'P53_109', 'P53_11', 'P53_110', 'P53_111', 'P53_112', 'P53_113',
  'P53_114', 'P53_115', 'P53_116', 'P53_117', 'P53_119', 'P53_12',
  'P53_120', 'P53_121', 'P53_13', 'P53_14', 'P53_15', 'P53_16',
  'P53_17', 'P53_18', 'P53_19', 'P53_2', 'P53_20', 'P53_21',
  'P53_22', 'P53_23', 'P53_24', 'P53_25', 'P53_26', 'P53_27',
  'P53_28', 'P53_29', 'P53_3', 'P53_30', 'P53_31', 'P53_32',
  'P53_33', 'P53_34', 'P53_35', 'P53_36', 'P53_37', 'P53_38',
  'P53_39', 'P53_4', 'P53_40', 'P53_41', 'P53_42', 'P53_43',
  'P53_44', 'P53_45', 'P53_46', 'P53_47', 'P53_48', 'P53_49',
  'P53_5', 'P53_50', 'P53_51', 'P53_52', 'P53_53', 'P53_54',
  'P53_55', 'P53_56', 'P53_57', 'P53_58', 'P53_59', 'P53_6',
  'P53_60', 'P53_61', 'P53_62', 'P53_63', 'P53_64', 'P53_65',
  'P53_66', 'P53_67', 'P53_68', 'P53_69', 'P53_7', 'P53_70',
  'P53_71', 'P53_72', 'P53_73', 'P53_74', 'P53_75', 'P53_76',
  'P53_77', 'P53_78', 'P53_79', 'P53_8', 'P53_80', 'P53_81',
  'P53_82', 'P53_83', 'P53_84', 'P53_85', 'P53_86', 'P53_87',
  'P53_88', 'P53_89', 'P53_9', 'P53_90', 'P53_91', 'P53_92',
  'P53_93', 'P53_94', 'P53_95', 'P53_96', 'P53_97', 'P53_98',
  'P53_99', 'WT_0', 'WT_1', 'WT_10', 'WT_100', 'WT_101', 'WT_102',
  'WT_103', 'WT_104', 'WT_105', 'WT_106', 'WT_107', 'WT_108',
  'WT_109', 'WT_11', 'WT_110', 'WT_111', 'WT_112', 'WT_113',
  'WT_114', 'WT_115', 'WT_116', 'WT_117', 'WT_118', 'WT_119',
  'WT_12', 'WT_121', 'WT_13', 'WT_14', 'WT_15', 'WT_16', 'WT_17',
  'WT_18', 'WT_19', 'WT_2', 'WT_20', 'WT_21', 'WT_22', 'WT_23',
  'WT_24', 'WT_25', 'WT_26', 'WT_27', 'WT_28', 'WT_29', 'WT_3',
  'WT_30', 'WT_31', 'WT_32', 'WT_33', 'WT_34', 'WT_35', 'WT_36',
  'WT_37', 'WT_38', 'WT_39', 'WT_4', 'WT_40', 'WT_41', 'WT_42',
  'WT_43', 'WT_44', 'WT_45', 'WT_46', 'WT_47', 'WT_48', 'WT_49',
  'WT_5', 'WT_50', 'WT_51', 'WT_52', 'WT_53', 'WT_54', 'WT_55',
  'WT_56', 'WT_57', 'WT_58', 'WT_59', 'WT_6', 'WT_60', 'WT_61',
  'WT_62', 'WT_63', 'WT_64', 'WT_65', 'WT_66', 'WT_67', 'WT_68',
  'WT_69', 'WT_7', 'WT_70', 'WT_71', 'WT_72', 'WT_73', 'WT_74',
  'WT_75', 'WT_76', 'WT_77', 'WT_78', 'WT_79', 'WT_8', 'WT_80',
  'WT_81', 'WT_82', 'WT_83', 'WT_84', 'WT_85', 'WT_86', 'WT_87',
  'WT_88', 'WT_89', 'WT_9', 'WT_90', 'WT_91', 'WT_92', 'WT_93',
  'WT_94', 'WT_95', 'WT_96', 'WT_97', 'WT_98', 'WT_99']

In [ ]:
influe = np.load("./Result_NvP53/NvP53_AtacRna_KoWt_Final_20221008/Explain/influe_diff.SharedScaner.atac.npz")['data']
influe = influe.T.astype(float)

# influe = np.load("./Result_NvP53/NvP53_AtacRna_KoWt_Final_20221008/Explain/influe_diff.SharedScaner.rna.npz")['data']
# influe = influe.T.astype(float)
influe.shape

In [ ]:
# kernal_influence = pd.DataFrame(influe)
kernal_influence = pd.DataFrame(influe, index=cellanno_atac)
kernal_influence.columns = kernal_influence.columns.map(lambda x: 'Motif_' + str(x))
kernal_influence.shape, kernal_influence.iloc[:5,:4]

In [ ]:
# influence_pos
influence_pos = kernal_influence.copy()
# influence_pos[influence_pos < 0] = 0
# influence_pos = influence_pos.loc[:, influence_pos.sum(0)!=0]


In [ ]:
influence_pos = norm_scale_influe(influence_pos)
influence_pos.shape

In [ ]:
sns.clustermap(influence_pos.T, cmap='vlag', figsize=(10, 10))

In [ ]:
bin_influ = binary_influ(influence_pos, t=0.85)
bin_influ.shape

In [ ]:
sns.clustermap(bin_influ.T, cmap='Greys', figsize=(5, 5))

In [ ]:
motif_var = influence_pos.var(axis=0)
sns.boxplot(motif_var)

In [ ]:
sns.clustermap(influence_pos.T[motif_var>0.85], standard_scale='row',
               cmap='vlag', figsize=(5, 3))

In [ ]:
motif_anno = pd.read_csv("./Result_NvP53/NvP53_AtacRna_KoWt_Final_20221008/Explain/tomtom_SharedScaner_atac/filterScaner_anno.csv",
                        index_col=0)
motif_anno.Query_ID = motif_anno.Query_ID.str.replace("Filter", "Motif")
motif_anno.head()

In [ ]:
motif_anno = motif_anno.drop_duplicates(['Query_ID', 'gene_name'])
motif_anno.shape

In [ ]:
motif_tf_mat = motif_anno.set_index(['Query_ID', 'gene_name'])
motif_tf_mat['cnt'] = 1
motif_tf_mat = motif_tf_mat['cnt'].unstack(fill_value=0)

In [ ]:
sel = motif_tf_mat.index.intersection(influence_pos.columns)

In [ ]:
influence_pos = influence_pos[sel]
influence_pos

In [ ]:
motif_tf_mat = motif_tf_mat.loc[sel]
motif_tf_mat

In [ ]:
ct_tf_influe = influence_pos@motif_tf_mat
ct_tf_influe

In [ ]:
sns.clustermap(ct_tf_influe.T, cmap='vlag', figsize=(15, 15))

In [ ]:
ct_tf_influe.to_csv("./Result_NvP53/NvP53_AtacRna_KoWt_Final_20221008/Explain/ct_tf_influe.atac.csv")

In [ ]:
ct_tf_influe.shape

In [ ]:
bin_ct_tf_influe = binary_influ(ct_tf_influe, t=0.95)
bin_ct_tf_influe.shape

In [ ]:
sns.clustermap(bin_ct_tf_influe, cmap='Greys', figsize=(20, 10))

### Difference between KO/WT

In [ ]:
data = ct_tf_influe[ct_tf_influe.index.str.find("24")!=-1].T
data

In [ ]:
def add_identity(axes, *line_args, **line_kwargs):
    identity, = axes.plot([], [], *line_args, **line_kwargs)
    def callback(axes):
        low_x, high_x = axes.get_xlim()
        low_y, high_y = axes.get_ylim()
        low = max(low_x, low_y)
        high = min(high_x, high_y)
        identity.set_data([low, high], [low, high])
    callback(axes)
    axes.callbacks.connect('xlim_changed', callback)
    axes.callbacks.connect('ylim_changed', callback)
    return axes

def plot_influe(x, y, top_n=2):
    ax = sns.scatterplot(x, y)

    top_gx = x[np.argsort(x)][:top_n].index.values
    low_gx = x[np.argsort(x)][-top_n:].index.values
    top_gy = y[np.argsort(y)][:top_n].index.values
    low_gy = y[np.argsort(y)][-top_n:].index.values
    genes = np.unique(np.hstack([top_gx, low_gx, top_gy, low_gy]))
    for g in genes:
        ax.annotate(g, (x[g], y[g]))
    ax = add_identity(ax, color='r', ls='--')
    return ax

x=ct_tf_influe.loc["WT_9"]
y=ct_tf_influe.loc["P53_9"]
plot_influe(x, y)

In [ ]:
plt.figure(figsize=(16, 10))
plt.subplot(2,3,1)
x = ct_tf_influe.loc["WT_15"]
y = ct_tf_influe.loc["P53_15"]
plot_influe(x, y)

plt.subplot(2,3,2)
x=ct_tf_influe.loc["WT_3"]
y=ct_tf_influe.loc["P53_3"]
plot_influe(x, y)

plt.subplot(2,3,3)
x=ct_tf_influe.loc["WT_1"]
y=ct_tf_influe.loc["P53_1"]
plot_influe(x, y)

plt.subplot(2,3,4)
x=ct_tf_influe.loc["WT_24"]
y=ct_tf_influe.loc["P53_24"]
plot_influe(x, y)

plt.subplot(2,3,5)
x=ct_tf_influe.loc["WT_25"]
y=ct_tf_influe.loc["P53_25"]
plot_influe(x, y)

plt.subplot(2,3,6)
x=ct_tf_influe.loc["WT_9"]
y=ct_tf_influe.loc["P53_9"]
plot_influe(x, y)

plt.show()